In [1]:
# ============================================================
# WEEK 9 BBO NOTEBOOK
# Week 1–8 inputs and outputs embedded directly
# Scaling + Emergence-Aware Bayesian Optimisation
# ============================================================

import numpy as np
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# WEEK 1–8 INPUT DATA
# ============================================================

inputs_by_week = [
    # Week 1
    [
        np.array([0.211325, 0.788675]),
        np.array([0.723607, 0.276393]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.125000, 0.375000, 0.625000, 0.875000]),
        np.array([0.875000, 0.625000, 0.375000, 0.125000]),
        np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
        np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
        np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
    ],

    # Week 2
    [
        np.array([0.654299, 0.353479]),
        np.array([0.754299, 0.253479]),
        np.array([0.304299, 0.553479, 0.709691]),
        np.array([0.804299, 0.603479, 0.409691, 0.205016]),
        np.array([0.854299, 0.653479, 0.359691, 0.155016]),
        np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
        np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
        np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
    ],

    # Week 3
    [
        np.array([0.582812, 0.421077]),
        np.array([0.712865, 0.284413]),
        np.array([0.118496, 0.481282, 0.876608]),
        np.array([1.000000, 0.683447, 0.334333, 0.000000]),
        np.array([0.882245, 0.615032, 0.380358, 0.114494]),
        np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
        np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
        np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
    ],

    # Week 4
    [
        np.array([0.183704, 0.127166]),
        np.array([0.255353, 0.749841]),
        np.array([0.094906, 0.770362, 0.859589]),
        np.array([0.820856, 0.226975, 0.784625, 0.113609]),
        np.array([0.917860, 0.273128, 0.475575, 0.052446]),
        np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
        np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
        np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
    ],

    # Week 5
    [
        np.array([0.582812, 0.421077]),
        np.array([0.684812, 0.281383]),
        np.array([0.171685, 0.508728, 0.819757]),
        np.array([0.734042, 0.687625, 0.466427, 0.303264]),
        np.array([0.568707, 0.909615, 0.957143, 0.011420]),
        np.array([0.744968, 0.084644, 0.801527, 0.169728, 0.990511]),
        np.array([1.000000, 0.229371, 0.690054, 0.244152, 0.521424, 1.000000]),
        np.array([0.022620, 0.000000, 0.288709, 0.762817, 0.551656, 0.961269, 0.843087, 1.000000])
    ],

    # Week 6
    [
        np.array([0.582812, 0.421077]),
        np.array([0.996744, 0.999561]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.743137, 0.718436, 0.581810, 0.365984]),
        np.array([0.572707, 0.914615, 0.952143, 0.016420]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.024461, 0.059677, 0.995330, 0.299568, 0.038576, 0.691027]),
        np.array([0.148189, 0.189483, 0.270757, 0.392507, 0.446351, 0.670593, 0.665417, 1.000000])
    ],

    # Week 7
    [
        np.array([0.582812, 0.421077]),
        np.array([0.744767, 0.322712]),
        np.array([0.167431, 0.503246, 0.833129]),
        np.array([0.692844, 0.522126, 0.400511, 0.337375]),
        np.array([0.543363, 0.994173, 0.995000, 0.003824]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.194327, 0.008117, 0.940397, 0.331469, 0.120428, 0.708672]),
        np.array([0.208946, 0.148993, 0.202639, 0.316908, 0.431312, 0.503474, 0.634079, 0.995000])
    ],

    # Week 8
    [
        np.array([0.582812, 0.421077]),
        np.array([0.723607, 0.276393]),
        np.array([0.173580, 0.509985, 0.861627]),
        np.array([0.605953, 0.443908, 0.321881, 0.510101]),
        np.array([0.503541, 0.995000, 0.995000, 0.000001]),
        np.array([0.339419, 0.513193, 0.599128, 0.825009, 0.789136]),
        np.array([0.232309, 0.000001, 0.881882, 0.376082, 0.191527, 0.731904]),
        np.array([0.045850, 0.409304, 0.057579, 0.179582, 0.499647, 0.251108, 0.757574, 0.939082])
    ]
]

# ============================================================
# WEEK 1–8 OUTPUT DATA
# ============================================================

outputs_by_week = [
    # Week 1
    np.array([
        1.1327056288856873e-125,
        0.5675786892822564,
        -0.032385408076210126,
        -17.20744048260943,
        60.223125,
        -1.3287857969718009,
        0.34356041660314957,
        9.0501517903694
    ]),

    # Week 2
    np.array([
        1.1867858443665185e-41,
        0.2715258567299176,
        -0.1198581070659559,
        -13.082213230390916,
        50.179390256321376,
        -1.5080002951000964,
        0.31639485635485903,
        9.0219949134074
    ]),

    # Week 3
    np.array([
        7.99676981308551e-19,
        0.5213723465552891,
        -0.04726977098841539,
        -25.67625344929702,
        64.78245026474816,
        -1.7372122723701597,
        0.3507828450585503,
        9.0587238074074
    ]),

    # Week 4
    np.array([
        -2.410121917336285e-100,
        0.0244939290195959,
        -0.04207093453964322,
        -20.330739763644917,
        61.278397876784794,
        -2.4624737429462145,
        0.0634803557047261,
        8.275283250689899
    ]),

    # Week 5
    np.array([
        7.99676981308551e-19,
        0.4258127251524913,
        -0.06232295859499999,
        -12.496845976106417,
        942.2521944777988,
        -2.280900502246122,
        0.21828066675598462,
        8.427686115001
    ]),

    # Week 6
    np.array([
        7.99676981308551e-19,
        -0.004122974640885927,
        -0.032385408076210126,
        -14.192389833871584,
        943.6841142765069,
        -1.2257941580841207,
        0.4410410448155631,
        9.3346877420455
    ]),

    # Week 7
    np.array([
        7.99676981308551e-19,
        0.3799297936947829,
        -0.03240117791304375,
        -6.92709780967855,
        1723.425126888365,
        -1.2923495282910902,
        0.919847131115406,
        9.472143824862
    ]),

    # Week 8
    np.array([
        7.99676981308551e-19,
        0.5381696913344642,
        -0.02792717356049581,
        -4.634066674167709,
        1679.193994434957,
        -1.0832489053663208,
        1.3511467710792968,
        9.1699591109441
    ])
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_history(function_index):
    X = np.vstack([week[function_index] for week in inputs_by_week])
    y = np.array([week[function_index] for week in outputs_by_week])
    return X, y


def scaled_target(y):
    """
    Compresses extreme output jumps while preserving sign.
    This is important for Function 5 because it has very large gains.
    """
    return np.sign(y) * np.log1p(np.abs(y))


def detect_emergence(y):
    """
    Detects sudden performance breakthroughs.
    """
    best_so_far = np.maximum.accumulate(y)
    improvements = np.diff(best_so_far)

    if len(improvements) == 0:
        return False, 0.0

    max_jump = np.max(improvements)
    baseline = np.median(np.abs(y)) + 1e-9
    emergence_score = max_jump / baseline

    return emergence_score > 3.0, emergence_score


def train_bayesian_model(X, y):
    """
    Trains a Gaussian Process surrogate model.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    y_scaled = scaled_target(y)

    kernel = (
        ConstantKernel(1.0)
        * Matern(length_scale=1.0, nu=2.5)
        + WhiteKernel(noise_level=1e-5)
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=True,
        n_restarts_optimizer=15,
        random_state=42
    )

    model.fit(X_scaled, y_scaled)
    return model, scaler


def generate_candidates(X, y, function_index, n_local=12000, n_global=2500):
    dim = X.shape[1]

    order = np.argsort(y)[::-1]
    best_x = X[order[0]]
    second_x = X[order[1]]

    emergent, emergence_score = detect_emergence(y)

    # Function-specific search settings
    if function_index == 4:
        # Function 5: strong emergent behaviour, exploit tightly
        local_radius = 0.008
        second_radius = 0.018
        global_count = 300

    elif function_index in [6, 7]:
        # Functions 7 and 8: improving but not as explosive as Function 5
        local_radius = 0.030
        second_radius = 0.055
        global_count = 1000

    elif function_index == 3:
        # Function 4: improving but still negative
        local_radius = 0.040
        second_radius = 0.070
        global_count = 1200

    else:
        # Other functions: balanced exploration
        local_radius = 0.055
        second_radius = 0.085
        global_count = n_global

    local_best = np.clip(
        best_x + np.random.normal(0, local_radius, size=(n_local, dim)),
        0.000001,
        0.995000
    )

    local_second = np.clip(
        second_x + np.random.normal(0, second_radius, size=(n_local // 2, dim)),
        0.000001,
        0.995000
    )

    # Interpolation between the top two regions
    mix = np.random.beta(2, 1, size=(2000, 1))

    interpolation = np.clip(
        mix * best_x + (1 - mix) * second_x + np.random.normal(0, 0.015, size=(2000, dim)),
        0.000001,
        0.995000
    )

    global_search = np.random.uniform(
        0.000001,
        0.995000,
        size=(global_count, dim)
    )

    candidates = np.vstack([
        local_best,
        local_second,
        interpolation,
        global_search,
        X
    ])

    return candidates, best_x, emergent, emergence_score


def acquisition_function(mu, sigma, candidates, best_x, function_index, emergent):
    """
    Upper Confidence Bound acquisition:
    score = predicted mean + beta * uncertainty - distance penalty
    """
    distance = np.linalg.norm(candidates - best_x, axis=1)

    if function_index == 4:
        # Function 5: exploit breakthrough region tightly
        beta = 0.10
        distance_penalty = 0.25

    elif function_index in [6, 7]:
        # Functions 7 and 8: moderate uncertainty search
        beta = 0.70
        distance_penalty = 0.08

    elif function_index == 3:
        # Function 4: cautious improvement
        beta = 0.90
        distance_penalty = 0.06

    else:
        # Other uncertain functions
        beta = 1.40
        distance_penalty = 0.04

    return mu + beta * sigma - distance_penalty * distance


def propose_week9_input(function_index):
    X, y = get_history(function_index)

    model, scaler = train_bayesian_model(X, y)

    candidates, best_x, emergent, emergence_score = generate_candidates(
        X,
        y,
        function_index
    )

    candidates_scaled = scaler.transform(candidates)

    mu, sigma = model.predict(candidates_scaled, return_std=True)

    scores = acquisition_function(
        mu,
        sigma,
        candidates,
        best_x,
        function_index,
        emergent
    )

    best_candidate = candidates[np.argmax(scores)]

    return (
        np.round(best_candidate, 6),
        np.max(y),
        X[np.argmax(y)],
        emergent,
        emergence_score
    )


# ============================================================
# GENERATE WEEK 9 INPUTS
# ============================================================

week9_inputs = []

print("WEEK 9 BBO INPUT GENERATION")
print("===========================\n")

for function_index in range(8):
    candidate, best_output, best_input, emergent, emergence_score = propose_week9_input(function_index)

    week9_inputs.append(candidate)

    print(f"Function {function_index + 1}")
    print(f"Best historical output: {best_output}")
    print(f"Best historical input:  {np.round(best_input, 6)}")
    print(f"Emergence detected:     {emergent}")
    print(f"Emergence score:        {emergence_score:.4f}")
    print(f"Week 9 suggested input: {candidate}")
    print()

# ============================================================
# PORTAL-READY WEEK 9 FORMAT
# ============================================================

print("PORTAL-READY WEEK 9 INPUTS")
print("==========================\n")

for i, arr in enumerate(week9_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

WEEK 9 BBO INPUT GENERATION

Function 1
Best historical output: 7.99676981308551e-19
Best historical input:  [0.582812 0.421077]
Emergence detected:     False
Emergence score:        0.0000
Week 9 suggested input: [0.582812 0.421077]

Function 2
Best historical output: 0.5675786892822564
Best historical input:  [0.723607 0.276393]
Emergence detected:     False
Emergence score:        0.0000
Week 9 suggested input: [0.69434  0.207302]

Function 3
Best historical output: -0.02792717356049581
Best historical input:  [0.17358  0.509985 0.861627]
Emergence detected:     False
Emergence score:        0.1197
Week 9 suggested input: [0.147117 0.526235 0.849011]

Function 4
Best historical output: -4.634066674167709
Best historical input:  [0.605953 0.443908 0.321881 0.510101]
Emergence detected:     False
Emergence score:        0.4084
Week 9 suggested input: [0.524137 0.403433 0.22003  0.482874]

Function 5
Best historical output: 1723.425126888365
Best historical input:  [0.543363 0.994173 0